# NEURAL NETWORKS AND DEEP LEARNING

---
A.A. 2025/26 (6 CFU) - Aidin Attar, Annalisa Gallina, Jacopo Pegoraro
---

## Lab. 02 - Regression and Classification with PyTorch

In this lab we move directly from data to trainable models. Instead of spending time on the closed-form least-squares derivation, we will start from the learning pipeline itself: data, model, loss, optimizer, validation, and generalization.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

## A few ideas to keep in mind

In this notebook we will use a simple regression problem to build a complete PyTorch workflow, from data generation to training and evaluation. The example is intentionally simple, but the questions behind it are the same ones that appear in much larger machine learning problems.

As you work through the lab, keep an eye on the relationship between training and validation performance. Fitting the training data well is important, but it is not enough: we also want the model to behave well on unseen data. This is why validation is useful when choosing hyper-parameters such as network size, learning rate, batch size, regularization, or number of epochs.

It is also worth paying attention to where the model is making predictions. Inside the range covered by the training samples, the network may behave reasonably well; outside that range, its output can become much less reliable. This will give us a concrete way to discuss model capacity, generalization, and the difference between interpolation and extrapolation.

# Part 1: Regression with PyTorch


In this first example we work on a simple one-dimensional regression problem.  
The setting is intentionally small enough to visualize clearly, so that we can focus on the training pipeline and on the behaviour of the model.

## Data generation

Let's generate some data with our usual polynomial model, and save the data points in two csv files, one for training (`train_data.csv`), and one for validation (`val_data.csv`).

You can find these files in the "Files" section of Colab, in the "content" folder. They are not stored in your local machine, but they are stored remotely in the Colab server.

In [ ]:
def poly_model(x, beta, noise_std=0):
    """
    INPUT
        x: x vector
        beta: polynomial parameters
        noise_std: enable noisy sampling (gaussian noise, zero mean, noise_std std)
    """
    pol_order = len(beta)
    x_matrix = np.array([x**i for i in range(pol_order)]).transpose()
    y = np.matmul(x_matrix, beta)
    noise = np.random.randn(len(y)) * noise_std
    return y + noise

beta_true = [-1.45, 1.12, 2.3]
noise_std = 0.2
np.random.seed(4)

### Train data
num_train_points = 20
x_train = np.random.rand(num_train_points)
y_train = poly_model(x_train, beta_true, noise_std)
with open('train_data.csv', 'w') as f:
    data = [f"{x},{y}" for x, y in zip(x_train, y_train)]
    f.write('\n'.join(data))

### Validation data
num_val_points = 20
x_val = np.random.rand(num_val_points)
y_val = poly_model(x_val, beta_true, noise_std)
with open('val_data.csv', 'w') as f:
    data = [f"{x},{y}" for x, y in zip(x_val, y_val)]
    f.write('\n'.join(data))


### Plot
plt.figure(figsize=(12,8))
x_highres = np.linspace(0,1,1000)
plt.plot(x_highres, poly_model(x_highres, beta_true), color='b', ls='--', label='True data model')
plt.plot(x_train, y_train, color='r', ls='', marker='.', label='Train data points')
plt.plot(x_val, y_val, color='g', ls='', marker='.', label='Validation data points')
plt.xlabel('x')
plt.ylabel('y')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

## Network Definition

Define a fully connected feed-forward network with 2 hidden layers.

Use a sigmoid activation function.

Since this is a regression, we do not want to limit the value of the output. For this reason, NO activation function should be used for the output layer.

In [ ]:
class Net(nn.Module):

    def __init__(self, Ni, Nh1, Nh2, No):
        """
        Ni - Input size
        Nh1 - Neurons in the 1st hidden layer
        Nh2 - Neurons in the 2nd hidden layer
        No - Output size
        """
        super().__init__()

        print('Network initialized')
        ###################################################
        ### PUT YOR CODE HERE
        self.fc1 = None
        self.fc2 = None
        self.out = None
        self.act = None
        ###################################################

    def forward(self, x, additional_out=False):
        ###################################################
        ### PUT YOR CODE HERE
        # don't forget the activations: not incuded by default!
        x = None
        ###################################################
        return x

## Dataset and Dataloader

We reuse the same `Dataset` pattern introduced in the previous lab. Here we simply wrap samples stored in a CSV file so that they can be fed to a PyTorch `DataLoader`.


### Dataset

In [ ]:
class CsvDataset(Dataset):

    def __init__(self, csv_file, transform=None):
        """
        Args:
            csv_file (string): Path to the csv file.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        self.transform = transform
        # Read the file and split the lines in a list
        with open(csv_file, 'r') as f:
            lines = f.read().split('\n')
        # Get x and y values from each line and append to self.data
        self.data = []
        for line in lines:
            sample = line.split(',')
            self.data.append((float(sample[0]), float(sample[1])))
        # Now self.data contains all our dataset.
        # Each element of the list self.data is a tuple: (input, output)

    def __len__(self):
        # The length of the dataset is simply the length of the self.data list
        return len(self.data)

    def __getitem__(self, idx):
        # Our sample is the element idx of the list self.data
        sample = self.data[idx]
        if self.transform:
            sample = self.transform(sample)
        return sample

### Transforms

In [ ]:
class ToTensor(object):
    """Convert sample to Tensors."""

    def __call__(self, sample):
        x, y = sample
        return (torch.tensor([x]).float(),
                torch.tensor([y]).float())

In [ ]:
composed_transform = transforms.Compose([ToTensor()])

train_dataset = CsvDataset('train_data.csv', transform=composed_transform)
val_dataset = CsvDataset('val_data.csv', transform=composed_transform)

### Dataloader

For the dataloader:

* enable the shuffling only for training data
* try different values for batch size
* disable the multiprocessing

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_dataloader  = DataLoader(val_dataset,  batch_size=len(val_dataset), shuffle=False, num_workers=0)

In this first example we work on a simple one-dimensional regression problem.  
The setting is intentionally small enough to visualize clearly, so that we can focus on the training pipeline and on the behaviour of the model.

## Training loop

Now we assemble the full PyTorch training pipeline: device selection, network initialization, loss, optimizer, training loop, and validation monitoring.


In [ ]:
# Check if the GPU is available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Training device: {device}")

In [ ]:
# Initialize the network
torch.manual_seed(0)
Ni = 1
Nh1 = 128
Nh2 = 256
No = 1
net = Net(Ni, Nh1, Nh2, No)
net.to(device)

In [ ]:
# Define the loss function
loss_fn = nn.MSELoss()

In [ ]:
# Define the optimizer
optimizer = optim.Adam(net.parameters(), lr=1e-3)

In [ ]:
### Configuration for animations and loss tracking
snapshot_freq = 50
train_loss_log = []
val_loss_log = []
animation_x = torch.linspace(0, 1, 400, device=device).unsqueeze(-1)
animation_x_np = animation_x.squeeze().cpu().numpy()
animation_true_model = poly_model(animation_x_np, beta_true).squeeze()
snapshot_epochs = [0]

net.eval()
# No gradients are needed during validation
# this saves memory and computation.
with torch.no_grad():
    prediction_history = [net(animation_x).squeeze().cpu().numpy()]

In [ ]:
### TRAINING LOOP ###
num_epochs = 3000
for epoch_num in range(num_epochs):
    print('#################')
    print(f'# EPOCH {epoch_num}')
    print('#################')

    ### TRAIN
    train_loss= []
    net.train() # Training mode (e.g. enable dropout, batchnorm updates,...)
    for sample_batched in train_dataloader:
        # Move data to device
        x_batch = sample_batched[0].to(device)
        label_batch = sample_batched[1].to(device)

        ###########################################
        ### PUT YOUR CODE HERE
        # Forward pass
        out = None

        # Compute loss
        loss = None

        # Backpropagation
        None

        # Update the weights
        None
        ###########################################

        # Save train loss for this batch
        loss_batch = loss.detach().cpu().numpy()
        train_loss.append(loss_batch)

    # Save average train loss
    train_loss = np.mean(train_loss)
    print(f"AVERAGE TRAIN LOSS: {train_loss}")
    train_loss_log.append(train_loss)

    ### VALIDATION
    val_loss= []
    net.eval() # Evaluation mode (e.g. disable dropout, batchnorm,...)
    with torch.no_grad(): # Disable gradient tracking
        for sample_batched in val_dataloader:
            # Move data to device
            x_batch = sample_batched[0].to(device)
            label_batch = sample_batched[1].to(device)

            ###########################################
            ### PUT YOUR CODE HERE
            # Forward pass
            out = None

            # Compute loss
            loss = None
            ###########################################

            # Save val loss for this batch
            loss_batch = loss.detach().cpu().numpy()
            val_loss.append(loss_batch)


        # Save average validation loss
        val_loss = np.mean(val_loss)
        print(f"AVERAGE VAL LOSS: {np.mean(val_loss)}")
        val_loss_log.append(val_loss)

    if epoch_num % snapshot_freq == 0 or epoch_num == num_epochs - 1:
        net.eval()
        with torch.no_grad():
            prediction_history.append(net(animation_x).squeeze().cpu().numpy())
        snapshot_epochs.append(epoch_num + 1)


### Plot losses

The loss curves give a compact summary of what happened during training.  
They are useful not only to check whether the model is learning, but also to spot possible issues such as underfitting, overfitting, or unstable optimization.

Additional tool for visualization:

*   [Weights and Biases](https://wandb.ai/site) (Suggested)
*   [Tensorboard](https://www.tensorflow.org/tensorboard?hl=it)



In [ ]:
# Plot losses
plt.figure(figsize=(12,8))
plt.semilogy(train_loss_log, label='Train loss')
plt.semilogy(val_loss_log, label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid()
plt.legend()
plt.show()

### Animate the training dynamics

The next cell replays how the regression curve changes during training. We store a prediction snapshot every `snapshot_freq` epochs and render it as an inline animation together with the corresponding loss curves.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

prediction_history_np = np.stack(prediction_history)
train_loss_arr = np.asarray(train_loss_log)
val_loss_arr = np.asarray(val_loss_log)
epoch_axis = np.arange(1, num_epochs + 1)

all_y_values = np.concatenate([
    prediction_history_np.reshape(-1),
    animation_true_model.reshape(-1),
    np.asarray(y_train).reshape(-1),
    np.asarray(y_val).reshape(-1),
])
y_min = all_y_values.min()
y_max = all_y_values.max()
y_margin = 0.05 * (y_max - y_min)
if y_margin == 0:
    y_margin = 1.0

loss_values = np.concatenate([train_loss_arr, val_loss_arr])
positive_loss_values = loss_values[loss_values > 0]
loss_min = positive_loss_values.min() * 0.8
loss_max = positive_loss_values.max() * 1.2

fig, (ax_model, ax_loss) = plt.subplots(2, 1, figsize=(12, 10))

ax_model.plot(animation_x_np, animation_true_model, '--', color='tab:blue', linewidth=2, label='True model')
ax_model.scatter(x_train, y_train, color='tab:red', s=35, label='Train data', zorder=3)
ax_model.scatter(x_val, y_val, color='tab:green', s=35, label='Validation data', zorder=3)
prediction_line, = ax_model.plot([], [], color='tab:orange', linewidth=3, label='Network prediction')
status_text = ax_model.text(
    0.02,
    0.98,
    '',
    transform=ax_model.transAxes,
    va='top',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
)
ax_model.set_xlim(animation_x_np.min(), animation_x_np.max())
ax_model.set_ylim(y_min - y_margin, y_max + y_margin)
ax_model.set_xlabel('x')
ax_model.set_ylabel('y')
ax_model.grid()
ax_model.legend(loc='best')

train_loss_line, = ax_loss.plot([], [], color='tab:red', linewidth=2, label='Train loss')
val_loss_line, = ax_loss.plot([], [], color='tab:green', linewidth=2, label='Validation loss')
ax_loss.set_xlim(0, num_epochs)
ax_loss.set_ylim(loss_min, loss_max)
ax_loss.set_yscale('log')
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss')
ax_loss.grid()
ax_loss.legend(loc='best')


def update(frame_idx):
    epoch = snapshot_epochs[frame_idx]
    prediction_line.set_data(animation_x_np, prediction_history_np[frame_idx])

    if epoch == 0:
        train_loss_line.set_data([], [])
        val_loss_line.set_data([], [])
        status_text.set_text('Epoch 0\nRandom initialization')
    else:
        train_loss_line.set_data(epoch_axis[:epoch], train_loss_arr[:epoch])
        val_loss_line.set_data(epoch_axis[:epoch], val_loss_arr[:epoch])
        status_text.set_text(
            f'Epoch {epoch}\n'
            f'Train loss: {train_loss_arr[epoch - 1]:.3e}\n'
            f'Val loss: {val_loss_arr[epoch - 1]:.3e}'
        )

    return prediction_line, train_loss_line, val_loss_line, status_text


anim = FuncAnimation(
    fig,
    update,
    frames=len(snapshot_epochs),
    interval=200,
    repeat=True,
    repeat_delay=1000,
)

plt.close(fig)
HTML(anim.to_jshtml())
# To export the animation as a GIF, uncomment the next line:
# anim.save('regression_training.gif', writer='pillow', fps=5)


## Network output

After training, we evaluate the network on a dense grid of input points.  
This gives a clearer picture of the function learned by the model than looking only at the training samples.

In [ ]:
# Input vector
x_vec = torch.linspace(0,1,1000)
x_vec = x_vec.to(device)
x_vec = x_vec.unsqueeze(-1)  # Adding a dimension to the input vector
print(f"Input shape: {x_vec.shape}")

# Network output
# eval() acts as switch for some specific layers/parts of the model that behave
# differently during training and inference (eval) time. For example, Dropout
# BatchNorm etc.
net.eval()
with torch.no_grad(): # turn off gradients computation
    y_vec = net(x_vec)
print(f"Output shape: {y_vec.shape}")

# Expected output
beta_true = [-1.45, 1.12, 2.3]
true_model = poly_model(x_vec.cpu().numpy(), beta_true).squeeze()

In [ ]:
# Convert x_vec and y_vec to numpy one dimensional arrays
x_vec = x_vec.squeeze().cpu().numpy()
y_vec = y_vec.squeeze().cpu().numpy()

# Plot output
plt.figure(figsize=(12,8))
plt.plot(x_vec, y_vec, label='Network output')
plt.plot(x_vec, true_model, label='True model')
plt.xlabel('x')
plt.ylabel('y')
plt.grid()
plt.legend()
plt.show()

### What if we predict outside the range?

A model can behave well on the region covered by the training data and still become unreliable outside it. This is the difference between interpolation and extrapolation.

In [ ]:
# Input vector
x_vec = torch.linspace(-5,5,1000)
x_vec = x_vec.to(device)
x_vec = x_vec.unsqueeze(-1)  # Adding a dimension to the input vector
print(f"Input shape: {x_vec.shape}")

# Network output
net.eval()
with torch.no_grad():
    y_vec = net(x_vec)
print(f"Output shape: {y_vec.shape}")

# Expected output
beta_true = [-1.45, 1.12, 2.3]
true_model = poly_model(x_vec.cpu().numpy(), beta_true).squeeze()

# Convert x_vec and y_vec to numpy one dimensional arrays
x_vec = x_vec.squeeze().cpu().numpy()
y_vec = y_vec.squeeze().cpu().numpy()

# Plot output
plt.figure(figsize=(12,8))
plt.plot(x_vec, y_vec, label='Network output')
plt.plot(x_vec, true_model, label='True model')
plt.xlabel('x')
plt.ylabel('y')
plt.grid()
plt.legend()
plt.show()

### What if we try different activation functions? (e.g., ReLU for nonlinearities)
### Is there overfit with ReLU?

### Homework : add some regularization to fight overfitting (e.g., dropout, weight decay,...)

### Activation-function animations

<img src="https://lh3.googleusercontent.com/d/1Lcd-6QtQYLX5v5uOZ2ycdqk5JPVD39ud=w900?v=4" width="720">

<img src="https://lh3.googleusercontent.com/d/1zx7wF9SVRBj9JlSVZO4QtFs7jWuzgYBu=w900?v=4" width="720">


## Access network parameters

Up to now we have looked at the model only from the outside, through its predictions.  
We now inspect its parameters directly to get a better feeling for what has been learned during training.


In [ ]:
# First hidden layer
h1_w = net.fc1.weight.data.cpu().numpy()
h1_b = net.fc1.bias.data.cpu().numpy()

# Second hidden layer
h2_w = net.fc2.weight.data.cpu().numpy()
h2_b = net.fc2.bias.data.cpu().numpy()

# Output layer
out_w = net.out.weight.data.cpu().numpy()
out_b = net.out.bias.data.cpu().numpy()

## Weights histogram

In [ ]:
# Weights histogram
fig, axs = plt.subplots(3, 1, figsize=(12,8))
axs[0].hist(h1_w.flatten(), 50)
axs[0].set_title('First hidden layer weights')
axs[1].hist(h2_w.flatten(), 50)
axs[1].set_title('Second hidden layer weights')
axs[2].hist(out_w.flatten(), 50)
axs[2].set_title('Output layer weights')
[ax.grid() for ax in axs]
plt.tight_layout()
plt.show()

#### Homework (simple): compare the histogram before and after the training

## Saving the network weights **(Essential for the Final Project)**

After training a model, a good practice is to save its weights in order to be able to retrieve it later.

**NOTE**: saving the weights is essential also for creating checkpoints of your training process. Google Colab imposes time limits on your kernel sessions, so *it will not be possible for you to carry out very long training sessions*.

It will be therefore essential for your final project to create checkpoints, so that you will be able to resume your training at a later moment, when re-gaining access of your Colab runtime. Even when using a dedicated GPU in your local machine or on a server, creating checkpoints is a good practice you should always implement in your training pipeline.

### Save network

In [ ]:
### Save network parameters
### Save the network state
# The state dictionary includes all the parameters of the network
net_state_dict = net.state_dict()
print(net_state_dict.keys())
# Save the state dict to a file
torch.save(net_state_dict, 'net_parameters.torch')

### Load network

In [ ]:
### Reload the network state
# First initialize the network (if not already done)
# IMPORTANT: you need to know the model definition!!
net = Net(Ni, Nh1, Nh2, No)
# Load the state dict previously saved
net_state_dict = torch.load('net_parameters.torch')
# Update the network parameters
net.load_state_dict(net_state_dict)

### Save optimizer state
Also the optimizer has its internal state!

You need to save both the network and the optimizer states if you want to continue your training.

If you are sure you have finished your training you can just save the network.

In [ ]:
### Save the optimizer state
torch.save(optimizer.state_dict(), 'optimizer_state.torch')

### Reload the optimizer state
optimizer = optim.Adam(net.parameters(), lr=0.001)
opt_state_dict = torch.load('optimizer_state.torch')
optimizer.load_state_dict(opt_state_dict)

## Analyze activations (Advanced)

In [ ]:
# First naive way: simply change the network definition to return an additional output

# More advanced strategy (optional if you're not familiar with Python!): using hooks

def get_activation(layer, input, output):
    global activation
    activation = torch.sigmoid(output)

### Register hook
hook_handle = net.fc2.register_forward_hook(get_activation)

### Analyze activations
net = net.to(device)
net.eval()
with torch.no_grad():
    x1 = torch.tensor([0.1]).float().to(device)
    y1 = net(x1)
    z1 = activation
    x2 = torch.tensor([0.9]).float().to(device)
    y2 = net(x2)
    z2 = activation
    x3 = torch.tensor([2.5]).float().to(device)
    y3 = net(x3)
    z3 = activation

### Remove hook
hook_handle.remove()

### Plot activations
fig, axs = plt.subplots(3, 1, figsize=(12,6))
axs[0].stem(z1.cpu().numpy())
axs[0].set_title('Last layer activations for input x=%.2f' % x1)
axs[1].stem(z2.cpu().numpy())
axs[1].set_title('Last layer activations for input x=%.2f' % x2)
axs[2].stem(z3.cpu().numpy())
axs[2].set_title('Last layer activations for input x=%.2f' % x3)
plt.tight_layout()
plt.show()

# Part 2: Classification model with PyTorch

We now move from regression to binary classification.  
The workflow remains very similar, but the goal changes: instead of predicting a continuous value, the network must decide which class each point belongs to.


**HINTS**
- Choose a loss function that is suitable for the specific problem, a binary classification in this case. If you keep a single linear output you can use a BCEWithLogitsLoss, which is more numerically stable than manually using a sigmoid output activation (more info [here](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)).
- The network now has 2 inputs. A batched input should have a shape `batch_size` $\times 2$.
- The dataset should be adapted accordingly. Also consider to increase the batch size.
- Explore different [optimizers](https://pytorch.org/docs/stable/optim.html), trying to understand the differences and their parameters.
- Try to increase the complexity of the network, and at the same time to introduce some regularization with dropout layers and/or weight decay (which is equivalent to an L2 regularization, typically implemented by the optimizer).
- Experiment with different hyper-parameters trying to minimize the VALIDATION LOSS. Once you are happy with the result, try the final test with the TEST dataset.

## Data generation

To build a classification dataset, we assign a label to each point in the plane according to a geometric rule.  
This gives us a synthetic problem that is easy to visualize and useful for testing how well a neural network can learn a nonlinear decision boundary.


In [ ]:
import itertools

np.random.seed(123)

def ellipse_mask(x1, x2, cx1, cx2, a, b):
    return ((x1 - cx1) / a) ** 2 + ((x2 - cx2) / b) ** 2 < 1

def bidimensional_model(x1, x2):
    out = ellipse_mask(x1, x2, 0, 0, 1, 1)
    out |= ellipse_mask(x1, x2, 5, 5, 2, 2)
    out |= ellipse_mask(x1, x2, -2.5, 5, 1, 2)
    out |= ellipse_mask(x1, x2, -6, -2.5, 3, 1)
    out |= ellipse_mask(x1, x2, -7.5, -5, 2, 4)
    out |= ellipse_mask(x1, x2, -7.5, 5, 6, 1)
    out |= ellipse_mask(x1, x2, 7.5, -7.5, 4, 4)
    out |= ellipse_mask(x1, x2, -1, -6, 3, 2)
    out |= ellipse_mask(x1, x2, 1, 6, 2, 5)
    out |= ellipse_mask(x1, x2, 6, 0, 2, 2)
    return out.astype(int)

### PLOT MODEL
# Input grid
x1 = np.linspace(-10, 10, 400)
x2 = np.linspace(-10, 10, 400)
x_prod = [x for x in itertools.product(x1, x2)]
x1 = np.array([x[0] for x in x_prod])
x2 = np.array([x[1] for x in x_prod])
# Evaluate out
y = bidimensional_model(x1, x2)
# Scatter plot
fig, ax = plt.subplots(figsize=(12,8))
colors = np.array(['C0', 'C1'])
ax.scatter(x1, x2, c=colors[y], s=10, marker='o')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
fig.show()

### Training points

In [ ]:
import pandas as pd

### Train data
num_points = 1000
x1 = np.random.uniform(-10, 10, num_points)
x2 = np.random.uniform(-10, 10, num_points)
y = bidimensional_model(x1, x2)
train_df = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y})
train_df.to_csv('classifier_train_data.csv', index=False)

fig, ax = plt.subplots(figsize=(12,8))
ax.scatter(x1, x2, c=colors[y], s=10, marker='o')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('Training points')
fig.show()

### Validation points

In this case: we generate validation points.
Usually: validation points are randomly selected from the training points (~20% of training data).

In [ ]:
### Validation data
num_points = 200
x1 = np.random.uniform(-10, 10, num_points)
x2 = np.random.uniform(-10, 10, num_points)
y = bidimensional_model(x1, x2)
val_df = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y})
val_df.to_csv('classifier_val_data.csv', index=False)

fig, ax = plt.subplots(figsize=(12,8))
ax.scatter(x1, x2, c=colors[y], s=10, marker='o')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('Validation points')
fig.show()

### Test points

In [ ]:
### Test data
num_points = 400
x1 = np.random.uniform(-10, 10, num_points)
x2 = np.random.uniform(-10, 10, num_points)
y = bidimensional_model(x1, x2)
val_df = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y})
val_df.to_csv('classifier_test_data.csv', index=False)

fig, ax = plt.subplots(figsize=(12,8))
ax.scatter(x1, x2, c=colors[y], s=10, marker='o')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('Test points')
fig.show()

## Dataset and Dataloader

The structure of the PyTorch pipeline stays the same as before, but the samples are now different: each input has two coordinates and each target is a binary label.

Define the dataset

In [ ]:
class ClassifierDataset(Dataset):

    def __init__(self, csv_file, transform=None):
        """
        Args:
            csv_file (string): Path to the csv file.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        ###############################
        # PUT YOUR CODE HERE
        self.transform = None
        # Read the file and store the content in a pandas DataFrame
        self.df = None
        ###############################

    def __len__(self):
        ###############################
        # PUT YOUR CODE HERE
        # The length of the dataset is simply the length of the self.df DataFrame
        return None
        ###############################

    def __getitem__(self, idx):
        # Our sample is the row at index idx of the dataframe

        ###############################
        # PUT YOUR CODE HERE
        row = self.df.iloc[idx]
        # There are 2 inputs this time
        sample = ([row.x1, row.x2], row.y)
        if self.transform:
            sample = None
        ###############################
        return sample

Define the transformations

In [ ]:
class ToTensor(object):
    """Convert sample to Tensors."""

    def __call__(self, sample):
        x, y = sample
        return (torch.Tensor(x).float(),
                torch.Tensor([y]).float())

Initialize the datasets

In [ ]:
###############################
### PUT YOUR CODE HERE
composed_transform = transforms.Compose([ToTensor()])

train_dataset = None
val_dataset   = None
test_dataset  = None
###############################

Define the dataloaders

In [ ]:
###############################
#Put your code here
train_dataloader = None #batch size = 200
val_dataloader   = None #The batch size can be equal to the validation set size if it fits in memory
test_dataloader  = None
###############################

## Network definition

In [ ]:
class Net(nn.Module):

    def __init__(self, Ni, Nh1, Nh2, No):
        """
        Ni - Input size
        Nh1 - Neurons in the 1st hidden layer
        Nh2 - Neurons in the 2nd hidden layer
        No - Output size
        """
        super().__init__()

        print('Network initialized')
        self.fc1 = nn.Linear(in_features=Ni, out_features=Nh1)
        self.fc2 = nn.Linear(in_features=Nh1, out_features=Nh2)
        self.out = nn.Linear(in_features=Nh2, out_features=No)
        self.act = nn.Sigmoid()

    def forward(self, x, additional_out=False):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.out(x)
        return x

## Training loop

The training procedure is conceptually the same as in regression.  
What changes are mainly the network input size, the loss function, and the interpretation of the output.


In [ ]:
# Initialize the network
torch.manual_seed(0)
Ni = 2
Nh1 = 128
Nh2 = 256
No = 1 #now we have classification between two classes

net = Net(Ni, Nh1, Nh2, No)
net.to(device)

# Define the loss function
###############################################
###PUT YOUR CODE HERE
loss_fn = None # TODO: use a proper classification loss

# Define the optimizer with lr=1e-2
optimizer = None
###############################################

In [ ]:
#### Configuration for animations and loss tracking
classification_snapshot_freq = 20
classification_grid_min = -10.0
classification_grid_max = 10.0
classification_grid_size = 150
classification_axis = np.linspace(classification_grid_min, classification_grid_max, classification_grid_size)
classification_grid_x1, classification_grid_x2 = np.meshgrid(classification_axis, classification_axis)
classification_animation_grid_np = np.column_stack([
    classification_grid_x1.ravel(),
    classification_grid_x2.ravel(),
])
classification_animation_grid = torch.tensor(classification_animation_grid_np).float().to(device)
train_loss_log = []
val_loss_log = []
classification_snapshot_epochs = [0]

net.eval()
with torch.no_grad():
    classification_snapshot_probs = [
        torch.sigmoid(net(classification_animation_grid))
        .reshape(classification_grid_size, classification_grid_size)
        .cpu()
        .numpy()
    ]

In [ ]:
### TRAINING LOOP
num_epochs = 600
for epoch_num in range(num_epochs):
    print('#################')
    print(f'# EPOCH {epoch_num}')
    print('#################')

    ### TRAIN
    train_loss= []
    ###############################################
    ###PUT YOUR CODE HERE
    net.train() # Training mode (e.g. enable dropout)
    for sample_batched in train_dataloader:
        # Move data to device
        x_batch = None
        label_batch = None

        # Forward pass
        out = None

        # Compute loss
        loss = None

        # Backpropagation
        None

        # Update the weights
        None

        # Save train loss for this batch
        loss_batch = loss.detach().cpu().numpy()
        train_loss.append(loss_batch)

    # Save average train loss
    train_loss = np.mean(train_loss)
    print(f"AVERAGE TRAIN LOSS: {train_loss}")
    train_loss_log.append(train_loss)

    ### VALIDATION
    val_loss= []
    net.eval() # Evaluation mode (e.g. disable dropout)
    with torch.no_grad(): # Disable gradient tracking
        for sample_batched in val_dataloader:
            # Move data to device
            x_batch = None
            label_batch = None

            # Forward pass
            out=None

            # Compute loss
            loss = None

            # Save val loss for this batch
            loss_batch = loss.detach().cpu().numpy()
            val_loss.append(loss_batch)
        ###############################################
        # Save average validation loss
        val_loss = np.mean(val_loss)
        print(f"AVERAGE VAL LOSS: {np.mean(val_loss)}")
        val_loss_log.append(val_loss)

    if epoch_num % classification_snapshot_freq == 0 or epoch_num == num_epochs - 1:
        net.eval()
        with torch.no_grad():
            classification_snapshot_probs.append(
                torch.sigmoid(net(classification_animation_grid))
                .reshape(classification_grid_size, classification_grid_size)
                .cpu()
                .numpy()
            )
        classification_snapshot_epochs.append(epoch_num + 1)


In [ ]:
# Plot losses
plt.figure(figsize=(12,8))
plt.semilogy(train_loss_log, label='Train loss')
plt.semilogy(val_loss_log, label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid()
plt.legend()
plt.show()

### Animate the decision boundary

The next cell replays how the decision boundary evolves during training. We store a prediction snapshot every `classification_snapshot_freq` epochs and render it as an inline animation together with the corresponding loss curves.


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

classification_snapshot_probs_np = np.stack(classification_snapshot_probs)
train_loss_arr = np.asarray(train_loss_log)
val_loss_arr = np.asarray(val_loss_log)
epoch_axis = np.arange(1, num_epochs + 1)

loss_values = np.concatenate([train_loss_arr, val_loss_arr])
positive_loss_values = loss_values[loss_values > 0]
loss_min = positive_loss_values.min() * 0.8
loss_max = positive_loss_values.max() * 1.2

prediction_cmap = LinearSegmentedColormap.from_list(
    'classification_prediction_map',
    ['#deebf7', '#f7f7f7', '#fdd0a2'],
)

fig, (ax_model, ax_loss) = plt.subplots(2, 1, figsize=(12, 10))

prob_image = ax_model.imshow(
    classification_snapshot_probs_np[0],
    origin='lower',
    extent=[
        classification_grid_min,
        classification_grid_max,
        classification_grid_min,
        classification_grid_max,
    ],
    cmap=prediction_cmap,
    vmin=0.0,
    vmax=1.0,
    alpha=0.95,
    aspect='equal',
)
ax_model.scatter(
    train_df['x1'],
    train_df['x2'],
    color='tab:red',
    s=18,
    alpha=0.7,
    label='Train data',
    zorder=3,
)
ax_model.scatter(
    val_df['x1'],
    val_df['x2'],
    color='tab:green',
    s=35,
    marker='x',
    linewidths=1.1,
    alpha=0.9,
    label='Validation data',
    zorder=3,
)
status_text = ax_model.text(
    0.02,
    0.98,
    '',
    transform=ax_model.transAxes,
    va='top',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
)
ax_model.set_xlim(classification_grid_min, classification_grid_max)
ax_model.set_ylim(classification_grid_min, classification_grid_max)
ax_model.set_xlabel('x1')
ax_model.set_ylabel('x2')
ax_model.grid()
ax_model.legend(
    handles=[
        Line2D([0], [0], marker='o', linestyle='', color='tab:red', markersize=7, label='Train data'),
        Line2D([0], [0], marker='x', linestyle='', color='tab:green', markersize=8, label='Validation data'),
        Line2D([0], [0], color='tab:orange', linewidth=3, label='Decision boundary'),
    ],
    loc='best',
)

train_loss_line, = ax_loss.plot([], [], color='tab:red', linewidth=2, label='Train loss')
val_loss_line, = ax_loss.plot([], [], color='tab:green', linewidth=2, label='Validation loss')
ax_loss.set_xlim(0, num_epochs)
ax_loss.set_ylim(loss_min, loss_max)
ax_loss.set_yscale('log')
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss')
ax_loss.grid()
ax_loss.legend(loc='best')

boundary_state = {'artist': None}


def update(frame_idx):
    frame_probs = classification_snapshot_probs_np[frame_idx]
    epoch = classification_snapshot_epochs[frame_idx]
    prob_image.set_data(frame_probs)

    if boundary_state['artist'] is not None:
        boundary_state['artist'].remove()
        boundary_state['artist'] = None

    if frame_probs.min() < 0.5 < frame_probs.max():
        boundary_state['artist'] = ax_model.contour(
            classification_grid_x1,
            classification_grid_x2,
            frame_probs,
            levels=[0.5],
            colors='tab:orange',
            linewidths=3,
        )

    if epoch == 0:
        train_loss_line.set_data([], [])
        val_loss_line.set_data([], [])
        status_text.set_text('Epoch 0\nRandom initialization')
    else:
        train_loss_line.set_data(epoch_axis[:epoch], train_loss_arr[:epoch])
        val_loss_line.set_data(epoch_axis[:epoch], val_loss_arr[:epoch])
        status_text.set_text(
            f'Epoch {epoch}\n'
            f'Train loss: {train_loss_arr[epoch - 1]:.3e}\n'
            f'Val loss: {val_loss_arr[epoch - 1]:.3e}'
        )

    return prob_image, train_loss_line, val_loss_line, status_text


anim = FuncAnimation(
    fig,
    update,
    frames=len(classification_snapshot_epochs),
    interval=200,
    repeat=True,
    repeat_delay=1000,
)

plt.close(fig)
HTML(anim.to_jshtml())
# anim.save('classification_training.gif', writer='pillow', fps=5)


## Final test

After training, we evaluate the model on a separate test set.  
This is the only stage where we use test data, so that the final accuracy provides an unbiased estimate of performance.

Iterate the dataloader a single time and save all the outputs (in case you have multiple batches)

In [ ]:
all_inputs = []
all_outputs = []
all_labels = []
net.eval() # Evaluation mode (e.g. disable dropout)
with torch.no_grad(): # Disable gradient tracking
    for sample_batched in test_dataloader:
        # Move data to device
        x_batch = sample_batched[0].to(device)
        label_batch = sample_batched[1].to(device)
        # Forward pass
        out = net(x_batch)
        # Save outputs and labels
        all_inputs.append(x_batch)
        all_outputs.append(out)
        all_labels.append(label_batch)
# Concatenate all the outputs and labels in a single tensor
all_inputs  = torch.cat(all_inputs)
all_outputs = torch.cat(all_outputs)
all_labels  = torch.cat(all_labels)

test_loss = loss_fn(all_outputs, all_labels)
print(f"AVERAGE TEST LOSS: {test_loss}")

In this case the network has a linear output (for a better stability of the loss function).
To have probability estimates you can apply a sigmoid to the network output.

Since we just need the most probable class and we have a single output, we can consider the sign of the linear output. Negative output means that the class 0 is the most probable (probability < 50%), otherwise class 1 (probability > 50%).

Essentially this network estimates the probability of the input sample to be of class 1.

> **NOTE**
>
> You can (and should, for practice) redefine the problem by defining a network with more than one output, each of them corresponding to a specific class (2 in this case). Since the two classes are mutually exclusive, the loss function should be a CrossEntropyLoss (softmax activation). In a multi-class scenario, a BCE loss is suitable when the classes are NOT mutually exclusive.






---


Get the most probable class inferred by the network


In [ ]:
# Get the most probable class inferred by the network
all_output_classes = torch.zeros(all_outputs.shape).to(device)
all_output_classes[all_outputs > 0] = 1

Evaluate the test accuracy

In [ ]:
tot_correct_out = (all_output_classes == all_labels).sum()
test_accuracy = 100 * tot_correct_out / len(all_labels)
print(f"TEST ACCURACY: {test_accuracy:.2f}%")

In [ ]:
len(all_output_classes)

Plot the results

In [ ]:
### Plot
x1 = all_inputs.squeeze().cpu().numpy()[:, 0]
x2 = all_inputs.squeeze().cpu().numpy()[:, 1]
y_true = all_labels.squeeze().cpu().numpy()
y_pred = all_output_classes.squeeze().cpu().numpy()

fig, ax = plt.subplots(figsize=(12,8))
# Plot predictions
ax.scatter(x1, x2, c=colors[y_pred.astype(np.uint8)], s=10, marker='o')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('Network Predictions')
# Mark wrong outputs
error_mask = y_pred != y_true
ax.scatter(x1[error_mask], x2[error_mask], color='red', s=100, marker='x', label='MISCLASSIFIED SAMPLES')
plt.legend()
fig.show()

# (Optional) High-level and Lightweight PyTorch wrapper: PyTorch Lightning


Lightning forces the following structure to your code which makes it reusable and shareable:

- Research code (the LightningModule).
- Engineering code (you delete, and is handled by the Trainer).
- Non-essential research code (logging, etc... this goes in Callbacks).
- Data (use PyTorch DataLoaders or organize them into a LightningDataModule).

Once you do this, you can train on multiple-GPUs, TPUs, CPUs and even in 16-bit precision without changing your code!

In [ ]:
! pip install pytorch-lightning

import os
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader, random_split
import pytorch_lightning as pl

In [ ]:
# Define a LightningModule (nn.Module subclass)
# A LightningModule defines a full system (ie: a GAN, autoencoder, BERT or a simple Image Classifier).
class LitNet(pl.LightningModule):
    def __init__(self, Ni, Nh1, Nh2, No):
        """
        Ni - Input size
        Nh1 - Neurons in the 1st hidden layer
        Nh2 - Neurons in the 2nd hidden layer
        No - Output size
        """
        super().__init__()

        print('Network initialized')
        self.net = nn.Sequential(nn.Linear(in_features=Ni, out_features=Nh1),
                       nn.Sigmoid(),
                       nn.Linear(in_features=Nh1, out_features=Nh2),
                       nn.Sigmoid(),
                       nn.Linear(in_features=Nh2, out_features=No))
        self.val_loss = []
        self.train_loss = []

    # Forward step defines how the LightningModule behaves during inference/prediction.
    def forward(self, x, additional_out=False):
        return self.net(x)

    # Training_step defines the training loop.
    def training_step(self, batch, batch_idx=None):
        # training_step defines the train loop. It is independent of forward
        x_batch = batch[0]
        label_batch = batch[1]
        out = self.net(x_batch)
        loss = F.binary_cross_entropy_with_logits(out, label_batch)
        self.train_loss.append(loss.item())
        return loss

    def validation_step(self, batch, batch_idx=None):
        # validation_step defines the validation loop. It is independent of forward
        x_batch = batch[0]
        label_batch = batch[1]
        out = self.net(x_batch)
        loss = F.binary_cross_entropy_with_logits(out, label_batch)
        self.val_loss.append(loss.item())
        self.log("val_loss", loss.item(), prog_bar=True)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.net.parameters(), lr=1e-2)
        return optimizer

In [ ]:
# TRAIN!
batch_size = 200
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader   = DataLoader(val_dataset,   batch_size=len(val_dataset), shuffle=False, num_workers=0)
test_dataloader  = DataLoader(test_dataset,  batch_size=len(test_dataset), shuffle=False, num_workers=0)

trainer = pl.Trainer(devices=1, max_epochs=20, val_check_interval=1)
litnet = LitNet(Ni, Nh1, Nh2, No)
trainer.fit(litnet, train_dataloader, val_dataloader)


In [ ]:
# Plot losses
plt.figure(figsize=(12,8))
plt.semilogy(litnet.train_loss, label='Train loss')
plt.semilogy(litnet.val_loss, label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid()
plt.legend()
plt.show()

# (Optional) Automatic Hyper-parameters Tuning: Optuna


Optuna is an automatic hyperparameter optimization software framework.

- Lightweight, versatile, and platform agnostic architecture
- Pythonic search spaces
- Efficient optimization algorithms
- Easy parallelization
- Quick visualization


Definitions:

- Study: optimization based on an objective function
- Trial: a single execution of the objective function

The goal of a study is to find out the optimal set of hyperparameter values (e.g., classifier and svm_c) through multiple trials (e.g., n_trials=100). Optuna is a framework designed for the automation and the acceleration of the optimization studies.

Optuna works with PyTorch, but also with Tensorflow, Keras and PyTorch Lightning! (and many others)

In [ ]:
! pip install optuna

In [ ]:
# Before we used:
# Nh1 = 128
# Nh2 = 256

import optuna

def objective(trial):

    # We optimize the number of hidden units in each layer.
    output_dims = [
        trial.suggest_int("n_units_l{}".format(i), 64, 256, log=True) for i in range(2)
    ]

    model = LitNet(Ni, output_dims[0], output_dims[1], No)

    trainer = pl.Trainer(devices=1, max_epochs=20, val_check_interval=1,
                         log_every_n_steps=1)
    hyperparameters = dict(output_dims=output_dims)
    trainer.logger.log_hyperparams(hyperparameters)
    trainer.fit(model, train_dataloader, val_dataloader)
    return trainer.callback_metrics["val_loss"].item()


pruner = optuna.pruners.NopPruner()
# print(pruner) <optuna.pruners._nop.NopPruner object at 0x7f4c2466ed50>
# print(type(pruner)) <class 'optuna.pruners._nop.NopPruner'>

study = optuna.create_study(study_name="myfirstoptimizationstudy", direction="minimize", pruner=pruner)
study.optimize(objective, n_trials=3, timeout=300)

print("Number of finished trials: {}".format(len(study.trials)))

print("Best trial:")
trial = study.best_trial

print("  Value: {}".format(trial.value))

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

# Homework

Try tuning other hyper-parameters, such as the learning rate.